In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json
from pathlib import Path
import logomaker
import string
import re
import glob
import matplotlib.lines as mlines

from matplotlib.ticker import FormatStrFormatter
from sklearn.metrics import adjusted_mutual_info_score as AMI
from matplotlib.patches import Patch
import nmi
from helps import *
fig, axs = plt.subplots(1, 2, figsize=(14, 1))
axs[0].bar(range(len(colors_to_use)), [1] * len(colors_to_use), color=colors_to_use)
axs[0].set_xticks(range(len(colors_to_use)))
axs[1].bar(range(len(colors_to_use_pastel)), [1] * len(colors_to_use_pastel), color=colors_to_use_pastel)
axs[1].set_xticks(range(len(colors_to_use_pastel)))
plt.show()

# SuppFig4

In [ ]:
datasets = ["PBMC","HSPC","BMMCMultiOme","MouseSkin","BMMCCite","Spleen"]

exp_map = {
    "Peak_mRNA_lncRNA": "Peak/ADT + mRNA + lncRNA",
    "ADT_mRNA_lncRNA": "Peak/ADT + mRNA + lncRNA",
    "Peak_mRNA": "Peak/ADT + mRNA",
    "ADT_mRNA": "Peak/ADT + mRNA",
    "Peak_lncRNA": "Peak/ADT + lncRNA",
    "ADT_lncRNA": "Peak/ADT + lncRNA",
    "Peak_GEX": "Peak/ADT + GEX",
    "ADT_GEX": "Peak/ADT + GEX",
    "mRNA_lncRNA": "mRNA + lncRNA",
}
legend_order = [
    "Peak/ADT + mRNA + lncRNA",
    "Peak/ADT + mRNA",
    "Peak/ADT + lncRNA",
    "Peak/ADT + GEX",
    "mRNA + lncRNA",
]
palette = {
    "Peak/ADT + mRNA + lncRNA": colors_to_use_pastel[4],
    "Peak/ADT + mRNA": colors_to_use_pastel[1],
    "Peak/ADT + lncRNA": colors_to_use_pastel[7],
    "Peak/ADT + GEX": colors_to_use_pastel[2],
    "mRNA + lncRNA": colors_to_use_pastel[8],
}

params = {
    "axes.labelsize": 17, "axes.titlesize": 17,
    "xtick.labelsize": 14, "ytick.labelsize": 14,
    "lines.linewidth": 3, "figure.dpi": 300
}
plt.rcParams.update(params)

fig, axs = plt.subplots(2, 3, figsize=(16, 10))
axs = axs.flatten()

for l, dataset in enumerate(datasets):
    df = pd.read_csv(f"Datasets/{dataset}/bionSBM/{dataset}_NMI.tsv.gz", sep="\t")
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed:")]
    df = df[df["GT"].eq("CellType")].copy()
    df["NMI/NMI*"] = df["NMI"] / df["NMI*"].replace(0, np.nan)
    df["log10(NMI/NMI*)"] = np.log10(df["NMI/NMI*"])
    df["Experiment"] = df["Exp"].str.replace("bionSBM_", "", regex=False).map(exp_map)
    df = df[df["Experiment"].notna()]

    for exp in legend_order:
        d = df[df["Experiment"].eq(exp)]
        if d.empty:
            continue
        stats = d.groupby("Level")["log10(NMI/NMI*)"].agg(["mean", "sem"]).reset_index()
        axs[l].errorbar(
            stats["Level"], stats["mean"], yerr=3*stats["sem"],
            marker="o", ms=6, capsize=3, color=palette[exp], ecolor=palette[exp],
            elinewidth=3, ls="-")
        
    axs[l].set_ylim([0, 3])
    axs[l].set_title(dataset, y=1.05)
    axs[l].set_xlabel("Hierarchy level")
    axs[l].set_ylabel("NMI/NMI* (log10)")
    axs[l].set_xticks(sorted(df["Level"].unique()))
    axs[l].spines["top"].set_visible(False)
    axs[l].spines["right"].set_visible(False)
    axs[l].annotate(string.ascii_uppercase[l], xy=(-0.18, 1.08), xycoords="axes fraction",
                    fontsize=20, fontweight="bold", ha="left")

handles = [Patch(facecolor=palette[k], edgecolor="0.4", linewidth=0.8) for k in legend_order]
fig.legend(handles, legend_order, title="Experiment", loc="center left", bbox_to_anchor=(1.02, 0.72),
           frameon=False, ncol=1, fontsize=14, title_fontsize=16)

fig.tight_layout(h_pad=3)
plt.savefig("Figures/bionSBM_SuppFig4.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig2

## Data

In [ ]:
metadata=pd.DataFrame()
shapes={}
for file in sorted(glob.glob("Datasets/*/CM/*GEX_Raw.h5ad")):
    adata=sc.read_h5ad(file)
    obs=pd.DataFrame(adata.obs["CellType"])
    dataset=file.split("/")[1]
    obs["Dataset"]=dataset
    shapes[dataset]=adata.shape
    metadata=pd.concat([metadata, obs])
with open('Tables/shapes.json', 'w') as f:
	json.dump(shapes, f)
metadata.to_csv("Tables/Metadata.tsv", sep="\t")

In [ ]:
data=pd.DataFrame(columns=["Dataset","Omic","Size"])
for omic in ["Peak","ADT","mRNA","lncRNA","GEX"]:
    print(omic)
    files=sorted(glob.glob(f"Datasets/*/CM/*{omic}_Def.h5ad"))
    for file in files:
        adata=sc.read_h5ad(file)
        ds=file.split("/")[1]
        d=pd.DataFrame([ds, omic, adata.shape[1]], index=data.columns).T
        data=pd.concat([data,d])

In [ ]:
data.to_csv("Tables/Metadata_features_size.tsv", sep="\t")

## Figure

In [ ]:
metadata=pd.read_csv("Tables/Metadata.tsv", sep="\t", index_col=0)
data=pd.read_csv("Tables/Metadata_features_size.tsv", sep="\t", index_col=0)
with open('Tables/shapes.json') as f:
	shapes = json.load(f)
cts={}
for ds in sorted(set(metadata["Dataset"])):
	d=metadata[metadata["Dataset"]==ds]
	cts[ds]=len(set(d["CellType"]))
mains=data[(data["Omic"]=="Peak") ^ (data["Omic"]=="ADT")][["Dataset","Size"]].set_index("Dataset")#.to_dict()["Size"]
gexs=data[data["Omic"]=="GEX"][["Dataset","Size"]].set_index("Dataset")#.to_dict()["Size"]
mrnas=data[data["Omic"]=="mRNA"][["Dataset","Size"]].set_index("Dataset")#.to_dict()["Size"]
lncrnas=data[data["Omic"]=="lncRNA"][["Dataset","Size"]].set_index("Dataset")#.to_dict()["Size"]

In [ ]:
tot=pd.concat([pd.DataFrame.from_dict(cts, orient="index"), mains,gexs, mrnas, lncrnas], axis=1)
tot.columns=["Number of cell types","Number of peaks or surface moelecules", "Number of genes","Number of mRNAs", "Number of lncRNAs"]
tot.to_csv("Tables/SuppTable1.tsv", sep="\t")
tot

In [ ]:
mains=data[(data["Omic"]=="Peak") ^ (data["Omic"]=="ADT")][["Dataset","Size"]].set_index("Dataset").to_dict()["Size"]
gexs=data[data["Omic"]=="GEX"][["Dataset","Size"]].set_index("Dataset").to_dict()["Size"]
mrnas=data[data["Omic"]=="mRNA"][["Dataset","Size"]].set_index("Dataset").to_dict()["Size"]
lncrnas=data[data["Omic"]=="lncRNA"][["Dataset","Size"]].set_index("Dataset").to_dict()["Size"]
params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axs = plt.subplots(1,3, dpi=300, figsize=(16,5))
axs=axs.flatten()

x=np.log10(np.array(list(shapes.values())).T[0])
y=[cts[key] for key in shapes.keys()]
axs[0].scatter(x=x, y=y)
for i, label in enumerate(shapes.keys()):
	axs[0].annotate(
		label, 
		(x[i], y[i]),			  # point position
		textcoords="offset points", 
		xytext=(-7,5),			  # offset text position
		ha='left', fontsize=11,
		arrowprops=dict(arrowstyle="->", color="gray", lw=0.5))
axs[0].set_xlabel("Number of cells [log10(count)]", size=13)
axs[0].set_ylabel("Number of cell types [count]", size=13)

x=np.log10(list(gexs.values()))
y=np.log10([mains[key] for key in gexs.keys()])
axs[1].scatter(x=x, y=y)
for i, label in enumerate(gexs.keys()):
	axs[1].annotate(
		label, 
		(x[i], y[i]),			  # point position
		textcoords="offset points", 
		xytext=(-7,5),			  # offset text position
		ha='left', fontsize=11,
		arrowprops=dict(arrowstyle="->", color="gray", lw=0.5))
axs[1].set_xlabel("Number of genes [log10(count)]", size=13)
axs[1].set_ylabel("Number of peaks or\nsurface proteins [log10(count)]", size=13)

x=np.log10(list(mrnas.values()))
y=np.log10([lncrnas[key] for key in mrnas.keys()])
axs[2].scatter(x=x, y=y)
for i, label in enumerate(mrnas.keys()):
	axs[2].annotate(
		label, 
		(x[i], y[i]),			  # point position
		textcoords="offset points", 
		xytext=(-7,5),			  # offset text position
		ha='left', fontsize=11,
		arrowprops=dict(arrowstyle="->", color="gray", lw=0.5))
axs[2].set_xlabel("Number of mRNAs [log10(count)]", size=13)
axs[2].set_ylabel("Number of lncRNAs [log10(count)]", size=13)

for i, ax in enumerate(axs):
	ax.spines['top'].set_visible(False)
	ax.spines['right'].set_visible(False)
	ax.annotate(string.ascii_uppercase[i],xy=(-0.25, 1.1),xycoords='axes fraction',fontsize=20,fontweight="bold",ha='left')

plt.tight_layout(w_pad=1.5)
plt.savefig("Figures/bionSBM_SuppFig2.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig3

In [ ]:
latent_dims, runs = [2, 6, 12, 18, 24], range(5)

metadata = {
	"PBMC": read_celltype("Datasets/PBMC/CM/PBMC_Peak_GEX_Def.h5mu"),
	"Spleen": read_celltype("Datasets/Spleen/CM/Spleen_ADT_GEX_Def.h5mu")}

methods = [
	("PBMC", "Mowgli", "Argmax", "factor", "Mowgli/LatentDim/LT_{lt}/Run{run}/MLD_Embedding.tsv.gz"),
	("PBMC", "ShareTopic", "Argmax", "topic", "ShareTopic/LatentDim/LT_{lt}/Run{run}/STLD_MaxTopic.tsv.gz"),
	("PBMC", "COBOLT", "Argmax", "factor", "COBOLT/LatentDim/LT_{lt}/Run{run}/STLD_Embedding.tsv.gz"),
	("PBMC", "MultiVI", "KMeans", "kmeans", "MultiVI/LatentDim/LT_{lt}/Run{run}/STLD_Embedding.tsv.gz"),
	("Spleen", "totalVI", "KMeans", "kmeans", "totalVI/LatentDim/LT_{lt}/Run{run}/STLD_Embedding.tsv.gz")]

rows = []
for dataset, method, assignment, kind, path in methods:
	for lt in latent_dims:
		for run in runs:
			print(dataset, method, lt, run)
			file = f"Datasets/{dataset}/" + path.format(lt=lt, run=run)
			if not os.path.isfile(file):
				print(dataset, method, lt, run, "NOT FOUND")
				continue

			clusters = {
				"factor": lambda: factor_labels(file),
				"topic": lambda: topic_labefor dataset, job in zip(["PBMC",,"MouseSkin", "HSPC","BMMCMultiOme"][2:],
                        ["Pb","Ms","Hs","Bm"][2:]):ls(file),
				"kmeans": lambda: kmeans_labels(file, metadata[dataset], run=run)}[kind]()

			observed, random_nmi, n_clusters = compute_nmi(metadata[dataset], clusters, seed=lt * 100 + run, n_shuffles=100)
			rows.append([dataset, lt, run, observed, random_nmi, method, assignment, n_clusters])

df = pd.DataFrame(rows, columns=["Dataset", "LT", "Run", "NMI", "NMI*", "Method", "Assignment", "N_clusters"])
df["NMI/NMI*"] = df["NMI"].div(df["NMI*"]).where(df["NMI*"] > 0)
df["log10(NMI/NMI*)"] = np.log10(df["NMI/NMI*"]).where(df["NMI/NMI*"] > 0)
df.sort_values(["Dataset", "Method", "LT", "Run"], inplace=True, ignore_index=True)

os.makedirs("Tables", exist_ok=True)
df.to_csv("Tables/NMI_vs_LatentDim.tsv.gz", sep="\t", compression="gzip", index=False)

## Figure

In [ ]:
df=pd.read_csv("Tables/NMI_vs_LatentDim.tsv.gz", sep="\t", index_col=0)
df["log(NMI*)"]=np.log10(df["NMI*"])
df.head()

In [ ]:
palette={alg : colors_to_use_pastel[i] for i,alg in enumerate(["ShareTopic","Mowgli","bionSBM","MultiVI","totalVI"])}
palette["COBOLT"]=colors_to_use_pastel[8]
del palette["bionSBM"]

params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axs = plt.subplots(1,3, figsize=(16,5))
axs=axs.flatten()

sns.stripplot(data=df, x="LT", y="log10(NMI/NMI*)", ax=axs[0], hue="Method", palette=palette, dodge=True, size=10)
sns.stripplot(data=df, x="LT", y="NMI", ax=axs[1], hue="Method", palette=palette, dodge=True, size=10)
sns.stripplot(data=df, x="LT", y="log(NMI*)", ax=axs[2], hue="Method", palette=palette, dodge=True, size=10)
for ax in axs:
	ax.legend().remove()

for l, label in enumerate(["log10(NMI/NMI*)","NMI", "log(NMI*)"]):
	mymax = 1 if label == "NMI" else np.max(df[label])+0.5
	mymin = 0 if label != "log(NMI*)" else np.min(df["log(NMI*)"])-0.25
	axs[l].vlines(x="12", ymin=mymin, ymax=mymax, color='red', zorder=1, linestyle="--")
	axs[l].spines['top'].set_visible(False)
	axs[l].spines['right'].set_visible(False)  
	axs[l].spines['bottom'].set_visible(True)
	axs[l].spines['left'].set_visible(True)  
	axs[l].set_xlabel("Number of topics")
	axs[l].annotate(string.ascii_uppercase[l],xy=(-0.25, 1.1),xycoords='axes fraction',fontsize=20,fontweight="bold",ha='left')
axs[0].set_ylim([0,3])
axs[1].set_ylim([0,1])
axs[2].set_ylim([np.min(df["log(NMI*)"])-0.25,-1])
desired_order = palette.keys()

handles = [
    mlines.Line2D([], [], color=palette[key], marker='o',
                  linestyle='None', markersize=10, label=key)
    for key in desired_order
]

axs[2].legend(
    handles=handles,
    bbox_to_anchor=(1.7, 0.75),
    title="Method", frameon=False,
    fontsize=14, title_fontsize=16,
    
)
plt.tight_layout(w_pad=1.5)
plt.savefig("Figures/bionSBM_SuppFig3.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig1

In [ ]:
nsb=pd.read_csv("Tables/Time_bionSBM_vs_nSBM_cells.tsv", sep="\t")
bio=pd.read_csv("Tables/Time_bionSBM_cells.tsv", sep="\t")
mow=pd.read_csv("Tables/Time_Mowgli_cells.tsv", sep="\t")
cob=pd.read_csv("Tables/Time_COBOLT_cells.tsv", sep="\t")
mul=pd.read_csv("Tables/Time_MultiVI_cells.tsv", sep="\t")
share=pd.read_csv("Tables/Time_ShareTopic_cells.tsv", sep="\t")

In [ ]:
palette = {alg: colors_to_use_pastel[i] for i, alg in enumerate(["ShareTopic", "Mowgli", "bionSBM", "MultiVI", "totalVI"])}
palette.update(COBOLT=colors_to_use_pastel[8], nSBM=colors_to_use_pastel[9])

plt.rcParams.update({"axes.labelsize": 17, "axes.titlesize": 17,
                     "xtick.labelsize": 14, "ytick.labelsize": 14,
                     "lines.linewidth": 3, "figure.dpi": 300})

fig, axs = plt.subplots(2, 2, figsize=(16, 10))
axs = axs.ravel()

for col, label in [("t_bionsbm", "bionSBM"), ("t_nsbm", "nSBM")]:
    axs[0].scatter(nsb["n_cells"], nsb[col], label=label, color=palette[label])
axs[0].set(ylabel="Graph building time [s]")

for col, label in [("ram_bionsbm", "bionSBM"), ("ram_nsbm", "nSBM")]:
    axs[1].scatter(nsb["n_cells"], nsb[col], label=label, color=palette[label])
axs[1].set_ylabel("Graph building RAM [GB]")

methods = [
    (mow, "Mowgli", "t_mowgli", "ram_mowgli"),
    (bio, "bionSBM", "t_bionsbm", "ram_bionsbm"),
    (share, "ShareTopic", "t_sharetopic", "ram_sharetopic"),
    (cob, "COBOLT", "t_cobolt", "ram_cobolt"),
    (mul, "MultiVI", "t_multivi", "ram_multivi"),
]

for df, label, time_col, ram_col in methods:
    axs[2].scatter(df["n_cells"], df[time_col] / 60, label=label, color=palette[label])
    axs[3].scatter(df["n_cells"], df[ram_col], label=label, color=palette[label])

axs[2].set_ylabel("Running time [min]")
axs[3].set_ylabel("Memory requirement [GB]")


for ax in (axs[1], axs[3]):
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper left", bbox_to_anchor=(1.05, 1.1), frameon=False,
             title="Method", fontsize=14, title_fontsize=16,
              labelcolor=[palette[label] for label in labels])

for i, ax in enumerate(axs):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("Number of cells")
    ax.annotate(string.ascii_uppercase[i], (-0.25, 1.1), xycoords="axes fraction",
                fontsize=20, fontweight="bold")

plt.tight_layout()
fig.savefig("Figures/bionSBM_SuppFig1.png", dpi=300, bbox_inches="tight")
plt.show()

# SuppFig9

In [ ]:
metrics_main={}
for dataset in ["PBMC","HSPC","MouseSkin","BMMCMultiOme","BMMCCite","Spleen"]:
    main_fs = "Peak" if dataset not in ["BMMCCite", "Spleen"] else "ADT"
    file=f"Datasets/{dataset}/bionSBM/DominantTopicStability_top3_cosine_{main_fs}.tsv"
    if os.path.isfile(file):
        d=pd.read_csv(file, index_col=0, sep="\t")
        d.index=d.index.str.replace("_","+")
        d.columns=d.columns.str.replace("_","+")
        metrics_main[dataset]=d
    else:
        print(file, "is missing")

metrics_mrna={}
for dataset in ["PBMC","HSPC","MouseSkin","BMMCMultiOme","BMMCCite","Spleen"]:
    file=f"Datasets/{dataset}/bionSBM/DominantTopicStability_top3_cosine_mRNA.tsv"
    if os.path.isfile(file):
        d=pd.read_csv(file, index_col=0, sep="\t")
        d.index=d.index.str.replace("_","+")
        d.columns=d.columns.str.replace("_","+")
        metrics_mrna[dataset]=d
    else:
        print(file, "is missing")

In [ ]:
sns.set(style="ticks")
sns.set_style("white")
params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axs = plt.subplots(4,3, figsize=(16,25))
axs=axs.flatten()

for l, dataset in enumerate(["Spleen","PBMC","HSPC","BMMCCite","MouseSkin","BMMCMultiOme"]):
    if dataset in metrics_main.keys():
        bar=True if l in [2,5] else False
        if dataset in ["BMMCCite", "Spleen"]:
            order=["ADT", "ADT+GEX", "ADT+mRNA", "ADT+lncRNA", "ADT+mRNA+lncRNA"]
        else:
            order=["Peaks", "Peaks+GEX", "Peaks+mRNAs", "Peaks+lncRNAs", "Peaks+mRNAs+lncRNAs"]
        sns.heatmap(metrics_main[dataset], annot=True, cbar_kws={"label" : "cosine similarity"}, annot_kws={"size": 30},  
                    ax=axs[l], cbar=bar, vmin=0, vmax=1, fmt='.2f')
      #  yticks=["\n".join(a.split("+")) for a in order]
       # axs[l].set_yticks(axs[l].get_yticks(), yticks, size=20)
        #axs[l].set_xticks(axs[l].get_xticks(), yticks, rotation=45, size=20)
        axs[l].spines['top'].set_visible(False)
        axs[l].spines['right'].set_visible(False)  
        axs[l].spines['bottom'].set_visible(True)
        axs[l].spines['left'].set_visible(True)  
        axs[l].text(-0.5, 1.075, string.ascii_uppercase[l], transform=axs[l].transAxes, size=20, weight='bold',rotation=0)  
        axs[l].set_title(dataset, y=1.05)
    else:
        continue

for l, dataset in enumerate(["Spleen","PBMC","HSPC","BMMCCite","MouseSkin","BMMCMultiOme"]):
    l=l+6
    if dataset in metrics_mrna.keys():
        bar=True if l in [8,11] else False
        if dataset in ["BMMCCite", "Spleen"]:
            order=["mRNA", "ADTs+mRNAs", "mRNA+lncRNAs", "ADTs+mRNAs+lncRNAs"]
        else:
            order=["mRNA", "Peaks+mRNAs", "mRNA+lncRNAs", "Peaks+mRNAs+lncRNAs"]
        mask = np.triu(np.ones_like(metrics_mrna[dataset], dtype=bool), k=1)
        sns.heatmap(metrics_mrna[dataset], annot=True, cbar_kws={"label" : "cosine similarity"}, annot_kws={"size": 34},  
                    ax=axs[l], cbar=bar, vmin=0, vmax=1, fmt='.2f', mask=mask)
      #  yticks=["\n".join(a.split("+")) for a in order]
       # axs[l].set_yticks(axs[l].get_yticks(), yticks, size=20)
        #axs[l].set_xticks(axs[l].get_xticks(), yticks, rotation=45, size=20)
        axs[l].spines['top'].set_visible(False)
        axs[l].spines['right'].set_visible(False)  
        axs[l].spines['bottom'].set_visible(False)
        axs[l].spines['left'].set_visible(False)  
        axs[l].text(-0.5, 1.075, string.ascii_uppercase[l], transform=axs[l].transAxes, size=20, weight='bold',rotation=0)  
        axs[l].set_title(dataset, y=1.05)
    else:
        continue

fig.tight_layout(h_pad=2, w_pad=6)
plt.savefig("Figures/bionSBM_SuppFig8.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig10

In [ ]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
datasets = ["BMMCMultiOme", "HSPC", "MouseSkin", "PBMC"]
k = 3
outdir = "Figures"
os.makedirs(outdir, exist_ok=True)

peak_order = [
    "Peak", "Peak+GEX", "Peak+mRNA", "Peak+lncRNA",
    "Peak+mRNA+lncRNA",
]
mrna_order = ["mRNA+lncRNA", "Peak+mRNA", "Peak+mRNA+lncRNA"]

palette = {
    "BMMCMultiOme": colors_to_use[8],
    "HSPC": colors_to_use[10],
    "MouseSkin": colors_to_use[5],
    "PBMC": colors_to_use[7],
}

params = {
    "axes.labelsize": 17,
    "axes.titlesize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "lines.linewidth": 3,
    "figure.dpi": 300,
}
plt.rcParams.update(params)
sns.set(style="ticks")
sns.set_style("white")

# ------------------------------------------------------------
# Load one topic-feature analysis
# ------------------------------------------------------------
def load_pairwise(topic_feature):
    tables = []
    for dataset in datasets:
        path = (
            f"Datasets/{dataset}/bionSBM/"
            f"DominantTopicStability_top{k}_cosine_{topic_feature}_pairwise.tsv.gz"
        )
        if os.path.isfile(path):
            tables.append(pd.read_csv(path, sep="\t"))
        else:
            print(f"Missing: {path}", flush=True)

    if not tables:
        raise FileNotFoundError(
            f"No pairwise stability files found for {topic_feature} topics."
        )

    data = pd.concat(tables, ignore_index=True)
    data["Experiment"] = data["Experiment"].str.replace("_", "+", regex=False)
    return data


def summarise(data):
    by_celltype = (
        data.groupby(["Dataset", "Experiment", "CellType"], as_index=False)
            .agg(MedianCosine=("CosineSimilarity", "median"),
                 NPairs=("CosineSimilarity", "size"))
    )

    run1 = data[["Dataset", "Experiment", "Run1", "CosineSimilarity"]].rename(
        columns={"Run1": "Run"}
    )
    run2 = data[["Dataset", "Experiment", "Run2", "CosineSimilarity"]].rename(
        columns={"Run2": "Run"}
    )
    by_run = (
        pd.concat([run1, run2], ignore_index=True)
          .groupby(["Dataset", "Experiment", "Run"], as_index=False)
          .agg(MedianCosine=("CosineSimilarity", "median"),
               NPairs=("CosineSimilarity", "size"))
    )
    return by_celltype, by_run


def effect_range(data, covariate, topic_feature):
    out = (
        data.groupby(["Dataset", "Experiment"])["MedianCosine"]
            .agg(lambda x: x.max() - x.min())
            .rename("Magnitude")
            .reset_index()
    )
    out["TopicFeature"] = topic_feature
    out["Covariate"] = covariate
    return out


peak_celltype, peak_run = summarise(load_pairwise("Peak"))
mrna_celltype, mrna_run = summarise(load_pairwise("mRNA"))

# Save the two directly comparable variability measures.
effects = pd.concat([
    effect_range(peak_celltype, "Cell type", "Peak"),
    effect_range(peak_run, "Run", "Peak"),
    effect_range(mrna_celltype, "Cell type", "mRNA"),
    effect_range(mrna_run, "Run", "mRNA"),
], ignore_index=True)

# ------------------------------------------------------------
# Plot: rows = topic feature; columns = source of variability
# ------------------------------------------------------------
fig, axs = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
axs = axs.flatten()

panels = [
    (peak_celltype, peak_order, "Peak topics: variation across cell types"),
    (peak_run, peak_order, "Peak topics: variation across runs"),
    (mrna_celltype, mrna_order, "mRNA topics: variation across cell types"),
    (mrna_run, mrna_order, "mRNA topics: variation across runs"),
]

for l, (data, order, title) in enumerate(panels):
    sns.boxplot(
        data=data,
        x="Experiment",
        y="MedianCosine",
        hue="Dataset",
        order=order,
        hue_order=datasets,
        palette=palette,
        showfliers=False,
        ax=axs[l],
        zorder=10
    )

    axs[l].axhline(0.5, ls="--", lw=1, color="black", alpha=0.7, zorder=1)
    axs[l].set_ylim(0, 1)
    axs[l].set_title(title)
    axs[l].set_xlabel("Modality combination")
    axs[l].set_ylabel("Median pairwise cosine similarity" if l in [0, 2] else "")
    axs[l].tick_params(axis="x", rotation=0)
    axs[l].spines["top"].set_visible(False)
    axs[l].spines["right"].set_visible(False)
    axs[l].spines["bottom"].set_visible(True)
    axs[l].spines["left"].set_visible(True)
    axs[l].annotate(
        string.ascii_uppercase[l],
        xy=(-0.05, 1.07),
        xycoords="axes fraction",
        fontsize=20,
        fontweight="bold",
        ha="left",
    )

    if axs[l].get_legend() is not None:
        axs[l].get_legend().remove()

    labels = [tick.get_text().replace("Peak+", "Peak+\n", 1) for tick in axs[l].get_xticklabels()]
    axs[l].set_xticklabels(labels, rotation=0, ha="center")
    
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    ncol=1,
    title="Dataset",
    fontsize=14,
    title_fontsize=16,
    frameon=False
)

fig.tight_layout(h_pad=3, w_pad=2)
plt.savefig("Figures/bionSBM_SuppFig10.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig6

In [ ]:
validation_df = pd.DataFrame([
    ["PBMC", "B_naive", "Topic 10", "Topic 6", "PAX5", 22, 0, "Top on both", 156, "Datasets/PBMC/bionSBM/Motifs/Peak_mRNA_lncRNA/Run22/Topic_6/knownResults.html"],   
    ["BMMCMultiOme", "Proerythroblast", "Topic 7", "Topic 10", "KLF1",19, 3, "Top in all 3", 10, 'Datasets/BMMCMultiOme/bionSBM/Motifs/Peak_mRNA_lncRNA/Run19/Topic_10/knownResults.html'],                              
    ["BMMCMultiOme", "Normoblast", "Topic 7", "Topic 10", "KLF1",19, 3, "Top in all 3", 10, 'Datasets/BMMCMultiOme/bionSBM/Motifs/Peak_mRNA_lncRNA/Run19/Topic_10/knownResults.html']],                              
    columns=["Dataset", "CellType", "mRNA-Topic", "Peak-topic", "Gene in mRNA-Topic", "Run", "Level", "Note", "Motif_Number", "Motif_HTML"])
validation_df

In [ ]:
dna_colors = {
    "A": colors_to_use[19],
    "C": colors_to_use[14],
    "G": colors_to_use[2],
    "T": colors_to_use[5]
}

BASE_COLOR = colors_to_use[11]
HIGHLIGHT_COLOR = colors_to_use[16]

top_n = 5
params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axes = plt.subplots(3, 3, figsize=(16, 15), dpi=300, gridspec_kw={"width_ratios": [0.7, 0.7, 1]},constrained_layout=True)
for i, (_, row) in enumerate(validation_df.iterrows()):

    dataset = row["Dataset"]
    run = int(row["Run"])
    level = int(row["Level"])

    mrna_topic_num = int(re.search(r"\d+", row["mRNA-Topic"]).group())
    peak_topic_num = int(re.search(r"\d+", row["Peak-topic"]).group())

    cell_type_to_plot=row["CellType"]

    # ---------------------------------------------
    # Load data
    # ---------------------------------------------
    metadata = pd.read_csv(
        f"Datasets/{dataset}/{dataset}_Metadata.tsv.gz",
        sep="\t",
        index_col=0
    )

    base_path = f"Datasets/{dataset}/bionSBM/bionSBM_Peak_mRNA_lncRNA/Runs/Run{run}"

    mRNA_topics = pd.read_csv(
        os.path.join(base_path,
        f"bionSBM_Peak_mRNA_lncRNA_level_{level}_mRNA_topics_documents.tsv.gz"),
        sep="\t",
        index_col=0
    )

    Peak_topics = pd.read_csv(
        os.path.join(base_path,
        f"bionSBM_Peak_mRNA_lncRNA_level_{level}_Peak_topics_documents.tsv.gz"),
        sep="\t",
        index_col=0
    )

    # Normalize topic naming
    mRNA_topics.columns = mRNA_topics.columns.str.replace("topics", "topic")
    Peak_topics.columns = Peak_topics.columns.str.replace("topics", "topic")

    # Align cells
    common_cells = (
        metadata.index
        .intersection(mRNA_topics.index)
        .intersection(Peak_topics.index)
    )

    metadata = metadata.loc[common_cells]
    mRNA_topics = mRNA_topics.loc[common_cells]
    Peak_topics = Peak_topics.loc[common_cells]

    # Center matrices (same as your logic)
    mRNA_centered = mRNA_topics - mRNA_topics.mean(axis=0)
    Peak_centered = Peak_topics - Peak_topics.mean(axis=0)

    # ---------------------------------------------
    # Compute enrichment for selected celltypes
    # ---------------------------------------------
    mask_ct = metadata["CellType"].isin([cell_type_to_plot])

    mrna_enrichment = mRNA_centered.loc[mask_ct].mean(axis=0)
    peak_enrichment = Peak_centered.loc[mask_ct].mean(axis=0)

    # Validated topic names
    mrna_topic_name = [c for c in mRNA_topics.columns if c.endswith(f"_{mrna_topic_num}")][0]
    peak_topic_name = [c for c in Peak_topics.columns if c.endswith(f"_{peak_topic_num}")][0]
    # Top N topics
    mrna_top = mrna_enrichment.sort_values(ascending=False).head(top_n)
    peak_top = peak_enrichment.sort_values(ascending=False).head(top_n)
    
    # ---------------------------------------------
    # Column 1 — mRNA
    # ---------------------------------------------
    ax_m = axes[i, 0]
    bars = ax_m.bar(range(len(mrna_top)), mrna_top.values, color=BASE_COLOR)

    if mrna_topic_name in mrna_top.index:
        idx = list(mrna_top.index).index(mrna_topic_name)
        bars[idx].set_color(HIGHLIGHT_COLOR)

    ax_m.set_xticks(range(len(mrna_top)))
    ax_m.set_xticklabels(mrna_top.index.str.replace("A_t","A-t").str.replace("_"," "), rotation=45, ha="right", rotation_mode="anchor")
    ax_m.set_title(f"{cell_type_to_plot} — mRNAs")
    ax_m.set_ylabel("Pc(mRNA-topic | cell type)")
   # ax_m.text(-0.1, 1.1, string.ascii_uppercase[i], transform=ax_m.transAxes, fontsize=20, fontweight='bold')
    ax_m.annotate(string.ascii_uppercase[i],xy=(-0.25, 1.1),xycoords='axes fraction',fontsize=20,fontweight="bold",ha='left')

    # ---------------------------------------------
    # Column 2 — Peak
    # ---------------------------------------------
    ax_p = axes[i, 1]
    bars = ax_p.bar(range(len(peak_top)), peak_top.values, color=BASE_COLOR)

    if peak_topic_name in peak_top.index:
        idx = list(peak_top.index).index(peak_topic_name)
        bars[idx].set_color(HIGHLIGHT_COLOR)

    ax_p.set_xticks(range(len(peak_top)))
    ax_p.set_xticklabels(peak_top.index.str.replace("k_t","k-t").str.replace("_"," "), rotation=45, ha="right", rotation_mode="anchor")
    ax_p.set_title(f"{cell_type_to_plot} — peaks")
    ax_p.text(0.5, 1.2, dataset, transform=ax_p.transAxes, fontsize=16, ha="center", va="center")
    # ---------------------------------------------
    # Column 3 — Motif Logo
    # ---------------------------------------------
    motif_number = row["Motif_Number"]
    if isinstance(motif_number, list):
        motif_number = motif_number[0]

    motif_path = (
        Path(f"Datasets/{dataset}/bionSBM/Motifs/"
             f"Peak_mRNA_lncRNA/Run{run}/"
             f"Topic_{peak_topic_num}/knownResults")
        / f"known{motif_number}.motif"
    )

    motif_df = pd.read_csv(motif_path, sep="\t", header=None, names=["A", "C", "G", "T"],skiprows=1)

    ax_logo = axes[i, 2]
    logo = logomaker.Logo(motif_df,ax=ax_logo,color_scheme=dna_colors)

    logo.style_spines(visible=False)
    logo.style_xticks(fmt='%d', anchor=0)
    ax_logo.set_ylabel("")
    ax_logo.set_title(row["Gene in mRNA-Topic"])

axes=axes.flatten()
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)  

plt.tight_layout(h_pad=2)
plt.savefig(f"Figures/bionSBM_SuppFig6.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig7

In [ ]:
validation_df = pd.DataFrame([
    ["PBMC", ["B_memory","B_naive"], "Topic 10", "Topic 6", "PAX5", 22, 0, "Top on both", 156, "Datasets/PBMC/bionSBM/Motifs/Peak_mRNA_lncRNA/Run22/Topic_6/knownResults.html"],
    ["HSPC", "GMP", "Topic 4", "Topic 12", "IRF8", 5, 2, "-", 44, 'Datasets/HSPC/bionSBM/Motifs/Peak_mRNA_lncRNA/Run5/Topic_12/knownResults.html'],
    ["BMMCMultiOme", ["Erythroblast","Normoblast","Proerythroblast"], "Topic 7", "Topic 10", "KLF1",19, 3, "Top in all 3", 10, 'Datasets/BMMCMultiOme/bionSBM/Motifs/Peak_mRNA_lncRNA/Run19/Topic_10/knownResults.html']],                              
    columns=["Dataset", "CellType", "mRNA-Topic", "Peak-topic", "Gene in mRNA-Topic", "Run", "Level", "Note", "Motif_Number", "Motif_HTML"])

In [ ]:
dna_colors = {
    "A": colors_to_use[19],
    "C": colors_to_use[14],
    "G": colors_to_use[2],
    "T": colors_to_use[5]
}

BASE_COLOR = colors_to_use[11]
HIGHLIGHT_COLOR = colors_to_use[16]
params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axes = plt.subplots(3, 2, figsize=(16, 15), dpi=300)

for i, (_, row) in enumerate(validation_df.iterrows()):

    dataset = row["Dataset"]
    run = int(row["Run"])
    level = int(row["Level"])

    mrna_topic_num = int(re.search(r"\d+", row["mRNA-Topic"]).group())
    peak_topic_num = int(re.search(r"\d+", row["Peak-topic"]).group())

    cell_types_to_highlight = row["CellType"]
    
    if isinstance(cell_types_to_highlight, str):
        cell_types_to_highlight = [cell_types_to_highlight]
    # ---------------------------------------------
    # Load data
    # ---------------------------------------------
    metadata = pd.read_csv(
        f"Datasets/{dataset}/{dataset}_Metadata.tsv.gz",
        sep="\t",
        index_col=0
    )

    base_path = f"Datasets/{dataset}/bionSBM/bionSBM_Peak_mRNA_lncRNA/Runs/Run{run}"

    mRNA_topics = pd.read_csv(
        os.path.join(base_path,
        f"bionSBM_Peak_mRNA_lncRNA_level_{level}_mRNA_topics_documents.tsv.gz"),
        sep="\t",
        index_col=0
    )

    Peak_topics = pd.read_csv(
        os.path.join(base_path,
        f"bionSBM_Peak_mRNA_lncRNA_level_{level}_Peak_topics_documents.tsv.gz"),
        sep="\t",
        index_col=0
    )

    # Normalize topic naming
    mRNA_topics.columns = mRNA_topics.columns.str.replace("topics", "topic")
    Peak_topics.columns = Peak_topics.columns.str.replace("topics", "topic")

    # Align cells
    common_cells = (
        metadata.index
        .intersection(mRNA_topics.index)
        .intersection(Peak_topics.index)
    )

    metadata = metadata.loc[common_cells]
    mRNA_topics = mRNA_topics.loc[common_cells]
    Peak_topics = Peak_topics.loc[common_cells]

    # Center matrices (same as your logic)
    mRNA_centered = mRNA_topics - mRNA_topics.mean(axis=0)
    Peak_centered = Peak_topics - Peak_topics.mean(axis=0)

    # ---------------------------------------------
    # Compute Pc(topic | celltype) for ALL celltypes
    # ---------------------------------------------

    def compute_ct(doc_df):
        df = doc_df.copy()
        df["CellType"] = metadata["CellType"]
        return df.groupby("CellType").mean()

    mrna_ct = compute_ct(mRNA_centered)
    peak_ct = compute_ct(Peak_centered)

    # Validated topic names
    mrna_topic_name = [
        c for c in mRNA_topics.columns
        if c.endswith(f"_{mrna_topic_num}")
    ][0]

    peak_topic_name = [
        c for c in Peak_topics.columns
        if c.endswith(f"_{peak_topic_num}")
    ][0]

    # Values across ALL cell types
    mrna_values = mrna_ct[mrna_topic_name].sort_values(ascending=False)
    peak_values = peak_ct[peak_topic_name].sort_values(ascending=False)

    # ---------------------------------------------
    # Column 1 — mRNA
    # ---------------------------------------------
    ax_m = axes[i, 0]

    bars = ax_m.bar(range(len(mrna_values)), mrna_values.values, color=BASE_COLOR)

    # Highlight validated cell type
    for ct in cell_types_to_highlight:
        if ct in mrna_values.index:
            idx = list(mrna_values.index).index(ct)
            bars[idx].set_color(HIGHLIGHT_COLOR)

    ax_m.set_xticks(range(len(mrna_values)))
    ax_m.set_xticklabels(mrna_values.index.str.replace("_", " "),rotation=90)
    mrna_topic_name=mrna_topic_name.replace("A_t","A-t").replace("_"," ")
    ax_m.set_title(mrna_topic_name)
    ax_m.set_ylabel("Pc(topic | cell type)")
    ax_m.annotate(string.ascii_uppercase[i],xy=(-0.25, 1.1),xycoords='axes fraction',fontsize=20,fontweight="bold",ha='left')

    # ---------------------------------------------
    # Column 2 — Peak
    # ------------------------<---------------------
    ax_p = axes[i, 1]

    bars = ax_p.bar(range(len(peak_values)), peak_values.values, color=BASE_COLOR)

    for ct in cell_types_to_highlight:
        if ct in peak_values.index:
            idx = list(peak_values.index).index(ct)
            bars[idx].set_color(HIGHLIGHT_COLOR)

    ax_p.set_xticks(range(len(peak_values)))
    ax_p.set_xticklabels(peak_values.index.str.replace("_", " "),rotation=90)
    peak_topic_name=peak_topic_name.replace("k_t","k-t").replace("_"," ")
    ax_p.set_title(peak_topic_name)
    ax_p.set_ylabel("Pc(topic | cell type)")


plt.tight_layout()

for i, (_, row) in enumerate(validation_df.iterrows()):
    # Force Matplotlib to update axis positions
    fig.canvas.draw()

    pos_left = axes[i, 0].get_position()
    pos_right = axes[i, 1].get_position()

    x_center = (pos_left.x1 + pos_right.x0) / 2
    y_center = pos_left.y1 + 0.035

    fig.text(x_center, y_center, row["Dataset"], ha="center", va="center", fontsize=16)

axes=axes.flatten()
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)  

plt.tight_layout()
plt.savefig(f"Figures/bionSBM_SuppFig7.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig8

In [ ]:
cases = [
    ["PBMC",         22, 0, 10,  6],
    ["HSPC",          5, 2,  4, 12],
    ["BMMCMultiOme", 19, 3,  7, 10],
]
# dataset, run, level, mRNA topic, Peak topic

BASE_COLOR = colors_to_use[11]
HIGHLIGHT_COLOR = colors_to_use[16]

# Only used to distinguish clusters in the composition panel
CLUSTER_COLORS = [colors_to_use[5], colors_to_use_pastel[4]]

plt.rcParams.update({
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "lines.linewidth": 4,
    "figure.dpi": 300,
})

fig, axes = plt.subplots(
    3, 3, figsize=(22, 15), dpi=300,
    gridspec_kw={"width_ratios": [1, 1, 1.3]},
    constrained_layout=True
)

for i, (dataset, run, level, mrna_topic, peak_topic) in enumerate(cases):

    base = (
        f"Datasets/{dataset}/bionSBM/"
        f"bionSBM_Peak_mRNA_lncRNA/Runs/Run{run}"
    )

    meta = pd.read_csv(
        f"Datasets/{dataset}/{dataset}_Metadata.tsv.gz",
        sep="\t", index_col=0
    )

    mrna = pd.read_csv(
        f"{base}/bionSBM_Peak_mRNA_lncRNA_level_{level}_mRNA_topics_documents.tsv.gz",
        sep="\t", index_col=0
    )

    peak = pd.read_csv(
        f"{base}/bionSBM_Peak_mRNA_lncRNA_level_{level}_Peak_topics_documents.tsv.gz",
        sep="\t", index_col=0
    )

    clust = pd.read_csv(
        f"{base}/bionSBM_Peak_mRNA_lncRNA_level_{level}_clusters.tsv.gz",
        sep="\t", index_col=0
    )

    # ------------------------------------------------------------
    # Hard bionSBM cluster assignment
    # ------------------------------------------------------------

    clusters = pd.Series(
        np.argmax(clust.to_numpy(), axis=0),
        index=clust.columns,
        name="Cluster"
    )

    mrna.columns = mrna.columns.str.replace("topics", "topic")
    peak.columns = peak.columns.str.replace("topics", "topic")

    common = (
        meta.index
        .intersection(mrna.index)
        .intersection(peak.index)
        .intersection(clusters.index)
    )

    meta = meta.loc[common]
    mrna = mrna.loc[common]
    peak = peak.loc[common]
    clusters = clusters.loc[common]

    # Same centering as original topic plots
    mrna = mrna - mrna.mean(axis=0)
    peak = peak - peak.mean(axis=0)

    mname = next(c for c in mrna.columns if c.endswith(f"_{mrna_topic}"))
    pname = next(c for c in peak.columns if c.endswith(f"_{peak_topic}"))

    cluster_order = np.sort(clusters.unique())
    x = np.arange(len(cluster_order))
    labels = [f"C{c}" for c in cluster_order]

    mvals = (
        mrna.groupby(clusters)[mname]
        .mean()
        .reindex(cluster_order)
    )

    pvals = (
        peak.groupby(clusters)[pname]
        .mean()
        .reindex(cluster_order)
    )

    # Cluster maximally enriched in each modality
    m_cluster = mvals.idxmax()
    p_cluster = pvals.idxmax()

    selected_clusters = (
        [m_cluster]
        if m_cluster == p_cluster
        else [m_cluster, p_cluster]
    )

    # ============================================================
    # Left + middle: topic enrichment
    # ============================================================

    for ax, vals, topic, omic in [
        (axes[i, 0], mvals, mrna_topic, "mRNA"),
        (axes[i, 1], pvals, peak_topic, "Peak"),
    ]:

        colors = [
            HIGHLIGHT_COLOR if c in selected_clusters else BASE_COLOR
            for c in cluster_order
        ]

        ax.bar(x, vals, color=colors)
        ax.axhline(0, color="black", lw=0.8)

        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=90, ha="right")
        ax.set_xlabel("bionSBM cluster")
        ax.set_ylabel("Pc(topic | cluster)")
        ax.set_title(f"{dataset} — {omic} topic {topic}")

    axes[i, 0].text(
        -0.12, 1.08,
        string.ascii_uppercase[i],
        transform=axes[i, 0].transAxes,
        fontsize=20,
        fontweight="bold"
    )

    # ============================================================
    # Right: cell-type fraction within selected cluster(s)
    # ============================================================

    comp = (
        pd.crosstab(clusters, meta["CellType"])
        .reindex(selected_clusters, fill_value=0)
    )

    # Remove cell types absent from all selected clusters
    comp = comp.loc[:, comp.sum(axis=0) > 0]

    # Convert counts -> fraction within each cluster
    comp = comp.div(comp.sum(axis=1), axis=0)

    # Order cell types by overall abundance across selected clusters
    comp = comp.loc[:, comp.sum(axis=0).sort_values(ascending=False).index]

    ax = axes[i, 2]

    celltypes = comp.columns
    x2 = np.arange(len(celltypes))

    if len(selected_clusters) == 1:
        c = selected_clusters[0]

        ax.bar(
            x2,
            comp.loc[c].to_numpy(),
            width=0.75,
            color=CLUSTER_COLORS[0],
            label=f"C{c}"
        )

    else:
        width = 0.38

        for j, c in enumerate(selected_clusters):
            ax.bar(
                x2 + (j - 0.5) * width,
                comp.loc[c].to_numpy(),
                width=width,
                color=CLUSTER_COLORS[j],
                label=f"C{c}"
            )

    ax.set_xticks(x2)
    ax.set_xticklabels(celltypes, rotation=45, ha="right")

    ax.set_ylim(0, 1)
    ax.set_xlabel("Cell type")
    ax.set_ylabel("Fraction of cluster")
    ax.set_title(f"{dataset} — annotation composition")

    if len(selected_clusters) > 1:
        ax.legend(frameon=False, title="Cluster")

axes=axes.flatten()
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)  
    
plt.savefig(f"Figures/bionSBM_SuppFig8.png", format="png", dpi=300, bbox_inches='tight')
plt.show()

# SuppFig5

In [ ]:
DATASETS = ["PBMC", "HSPC", "MouseSkin", "BMMCMultiOme", "BMMCCite", "Spleen"]
ATAC = DATASETS[:4]

EXP_TYPES = ["mRNA_lncRNA", "mRNA", "lncRNA", "GEX"]
REG_TYPES = ["mRNA_lncRNA", "mRNA", "GEX"]
EXPS = [f"Peak_{x}" for x in EXP_TYPES]

ORDER = ["PBMC", "HSPC", "BMMCMultiOme", "MouseSkin", "Spleen", "BMMCCite"]
ATAC_ORDER = ORDER[:4]


def read(path, **cols):
    if not os.path.isfile(path):
        print(f"Missing: {path}")
        return
    return pd.read_csv(path, sep="\t").assign(**cols)


# dCorr
metrics = {e: {} for e in EXPS}
for e in EXPS:
    for d in ATAC:
        p = f"Datasets/{d}/bionSBM/{e}_dCorr_ATAC_GEX_results.tsv"
        if (x := read(p, dataset=d, exp=e)) is not None:
            metrics[e][d] = x


# TF coupling
all_data = {f"Peak_{s}": {} for s in REG_TYPES}
for e in all_data:
    for d in ATAC:
        p = f"Datasets/{d}/bionSBM/bionSBM_{e}/{e}_TF_coupling_ATAC_GEX_results_matrix.tsv"
        if (x := read(p, dataset=d, exp=e)) is not None:
            all_data[e][d] = x


# TF enrichment
tables = []
for d in DATASETS:
    fs = "Peak" if d in ATAC else "ADT"
    for s in REG_TYPES:
        e = f"{fs}_{s}"
        p = f"Datasets/{d}/bionSBM/bionSBM_{e}/{e}_TF_enrichment_results.tsv"
        if (x := read(p, dataset=d, exp=e, exp_type=s)) is not None:
            tables.append(x)

results_df = pd.concat(tables, ignore_index=True)

In [ ]:
COLORS = { "mRNA_lncRNA" : colors_to_use[5],
           "mRNA" : colors_to_use[1],
           "lncRNA" : colors_to_use[2],
           "GEX" : colors_to_use[0]}

LABELS = dict(zip(EXP_TYPES, [
    "Peak/ADT + mRNA + lncRNA",
    "Peak/ADT + mRNA",
    "Peak/ADT + lncRNA",
    "Peak/ADT + GEX",
]))

plt.rcParams.update({
    "axes.labelsize": 17, "axes.titlesize": 17,
    "xtick.labelsize": 14, "ytick.labelsize": 14,
    "lines.linewidth": 3, "figure.dpi": 300,
})


def violin(ax, df, x, xlabel, panel, order, hue_order=None,
           lines=(), xlim=None, xticks=None, panel_x=-.2):
    sns.violinplot(
        data=df, y="dataset", x=x, order=order,
        hue="exp_type" if hue_order else None, hue_order=hue_order,
        palette=COLORS if hue_order else None,
        color=None if hue_order else colors_to_use[3],
        dodge=True, cut=0, linewidth=.5, ax=ax
    )
    for v in lines:
        ax.axvline(v, ls="--", color="black", alpha=.8, lw=1.5)

    ax.set(xlabel=xlabel, ylabel="")
    if xlim: ax.set_xlim(*xlim)
    if xticks: ax.set_xticks(xticks)
    ax.spines[["top", "right"]].set_visible(False)
    ax.text(panel_x, 1.035, panel, transform=ax.transAxes,
            fontsize=20, fontweight="bold")

    h, l = ax.get_legend_handles_labels()
    if ax.legend_: ax.legend_.remove()
    return h, l


# ---------- Z-SCORE ROW ----------
A = pd.concat([
    df[["zscore"]].assign(dataset=d, exp_type=e.removeprefix("Peak_"))
    for e, ds in metrics.items() for d, df in ds.items()
], ignore_index=True)

A = (A.replace([np.inf, -np.inf], np.nan)
     .dropna(subset=["zscore"]).query("zscore > 0")
     .assign(log_zscore=lambda x: np.log10(x.zscore)))

B = pd.DataFrame([
    {
        "dataset": d,
        "exp_type": e.removeprefix("Peak_"),
        "tf_coupling": g.pivot_table(
            index="ATAC_topic", columns="GEX_topic",
            values="zscore", aggfunc="first"
        ).fillna(0).max(axis=1).mean()
    }
    for e, ds in all_data.items()
    for d, df in ds.items()
    for _, g in df.groupby("run", sort=True)
])

C = (results_df.replace([np.inf, -np.inf], np.nan)
     .dropna(subset=["enrichment"]).query("enrichment > 0")
     .assign(log_enrich=lambda x: np.log10(x.enrichment)))


# ---------- ENRICHMENT ROW ----------
D = pd.concat(metrics["Peak_mRNA_lncRNA"].values(), ignore_index=True).query("zscore > 0").copy()
D["enrichment"] = D.dcor_real / D.dcor_null_mean

E = pd.DataFrame([
    {
        "dataset": d,
        "enrichment": g.pivot(
            index="ATAC_topic", columns="GEX_topic",
            values="enrichment"
        ).fillna(0).max(axis=1).mean()
    }
    for d, df in all_data["Peak_mRNA_lncRNA"].items()
    for _, g in df.groupby("run", sort=True)
])

F = results_df.query("exp_type == 'mRNA_lncRNA' and enrichment > 0")


# ---------- PLOT ----------
fig, ax = plt.subplots(2, 3, figsize=(16, 10), dpi=300)

h, l = violin(
    ax[0, 0], A, "log_zscore",
    "Distance correlation\nZ-score real vs null model [log10]", "A",
    ATAC_ORDER, EXP_TYPES, lines=[1, 2, 3], xlim=(-2, 7), xticks=[0,1,2,3])

violin(
    ax[0, 1], B, "tf_coupling",
    "TF-mediated coupling\nZ-score real vs null model", "B",
    ATAC_ORDER, REG_TYPES, lines=[1, 2, 3],
    xlim=(-1, 8), xticks=[0, 1, 2, 3, 5, 7])

violin(
    ax[0, 2], C, "log_enrich",
    "TF-target enrichment\n[log10]", "C",
    ORDER, REG_TYPES, lines=[0], xlim=(-1.5, 1.6), panel_x=-.1)

violin(ax[1, 0], D, "enrichment", "Distance correlation\nreal-null model ratio", "D", ATAC_ORDER)
violin(ax[1, 1], E, "enrichment", "TF-mediated coupling\nreal-null model ratio", "E", ATAC_ORDER)
violin(ax[1, 2], F, "enrichment", "TF-target enrichment\nreal-null model ratio", "F", ORDER)

for a in ax[1]:
    a.set_xscale("log")

ax[0, 2].legend(
    h, [LABELS[x] for x in l],
    title="Experiment", loc="center left", bbox_to_anchor=(1.02, .5),
    frameon=False, fontsize=14, title_fontsize=16,
    handlelength=1.2, handletextpad=.4, borderpad=.3, labelspacing=.3
)

plt.tight_layout()
plt.savefig("Figures/bionSBM_SuppFig5.png", dpi=300, bbox_inches="tight")
plt.show()